<a href="https://colab.research.google.com/github/BerkKilincer-dev/SoftIto/blob/main/transformers-llm/05_modern_transformer_bilesenleri.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Modern Transformer Bileşenleri: LLaMA, Mistral ve Mixtral'in İçinde Ne Var?

`01_transformer_sifirdan` defterinde 2017'deki orijinal Transformer'ı yazdık. Bugünkü LLM'ler aynı iskeleti kullanıyor ama neredeyse her parçası değişti:

| Bileşen | Orijinal Transformer (2017) | Modern LLM (LLaMA-3, Mistral, Qwen...) |
|---|---|---|
| Dikkat hesabı | $\text{softmax}(QK^\top/\sqrt{d_k})V$, tam matris bellekte | Aynı formül, **FlashAttention** ile parça parça |
| Konum bilgisi | Sinüzoidal, girdiye **eklenir** | **RoPE**: Q ve K vektörleri **döndürülür** |
| Gelecek maskesi | Sadece decoder'da | **Causal mask** her katmanda |
| Normalizasyon | Post-LayerNorm | Pre-**RMSNorm** |
| İleri beslemeli ağ | ReLU MLP | **SwiGLU** |
| Ölçekleme | Yoğun (dense) | **Mixture of Experts** (Mixtral, DeepSeek, Qwen-MoE) |

Bu defterde her bileşeni sıfırdan yazıp PyTorch'un hazır versiyonlarıyla doğruluyoruz ve en sonunda hepsini tek bir **LLaMA tarzı blokta** birleştiriyoruz.

In [ ]:
import math
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

torch.manual_seed(42)
plt.rcParams["figure.dpi"] = 100
print("PyTorch:", torch.__version__)

---
## 1) Self-Attention ve Multi-Head Attention: hızlı tekrar

$$\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}}\right) V$$

- $Q = XW_Q$, $K = XW_K$, $V = XW_V$; her token bir **sorgu** (ne arıyorum?), **anahtar** (ne içeriyorum?) ve **değer** (ne aktaracağım?) üretir.
- $\sqrt{d_k}$'ya bölmek, skorların varyansını 1 civarında tutar. Bölmezsek softmax doygunluğa ulaşır ve gradyanlar söner.
- **Multi-Head:** $d_{\text{model}}$ boyutu $h$ kafaya bölünür, her kafa farklı bir ilişki tipini öğrenir, sonuçlar birleştirilip $W_O$ ile karıştırılır.

In [ ]:
def dikkat(Q, K, V, maske=None):
    d_k = Q.size(-1)
    skor = Q @ K.transpose(-2, -1) / math.sqrt(d_k)
    if maske is not None:
        skor = skor.masked_fill(~maske, float("-inf"))
    agirlik = F.softmax(skor, dim=-1)
    return agirlik @ V, agirlik

# √d_k ölçeklemesinin etkisi
d_k = 512
q, k = torch.randn(1000, d_k), torch.randn(1000, d_k)
ham = (q * k).sum(-1)
print(f"Ölçeklenmemiş q·k varyansı: {ham.var():.1f}  (≈ d_k = {d_k})")
print(f"q·k / √d_k varyansı        : {(ham / math.sqrt(d_k)).var():.2f}  (≈ 1)")

skorlar = torch.randn(1, 8) * math.sqrt(d_k)
print("\nÖlçeklenmemiş softmax:", F.softmax(skorlar, -1).numpy().round(3), "→ neredeyse one-hot, gradyan ≈ 0")
print("Ölçeklenmiş softmax  :", F.softmax(skorlar / math.sqrt(d_k), -1).numpy().round(3))

---
## 2) Causal (Nedensel) Maske: geleceği gizlemek

Bir dil modeli $P(x_t \mid x_{<t})$ öğrenir. Eğitimde tüm dizi aynı anda verildiği için, $t$ pozisyonundaki token $t+1, t+2, \dots$ tokenlarına bakabilirse **kopya çeker**: cevabı doğrudan görür. Causal maske, dikkat matrisinin üst üçgenini $-\infty$ yapar, softmax sonrası bu ağırlıklar **0** olur.

```
          anahtar →  Ben  bugün  okula  gittim
sorgu ↓   Ben       [ ✓     ✗      ✗      ✗ ]
          bugün     [ ✓     ✓      ✗      ✗ ]
          okula     [ ✓     ✓      ✓      ✗ ]
          gittim    [ ✓     ✓      ✓      ✓ ]
```

Bu sayede **tek bir ileri geçişte** $T$ farklı "sonraki token tahmini" problemi paralel olarak eğitilir.

In [ ]:
tokenlar = ["Ben", "bugün", "okula", "yürüyerek", "gittim"]
T, d = len(tokenlar), 16
x = torch.randn(T, d)
causal_maske = torch.tril(torch.ones(T, T, dtype=torch.bool))

_, maskesiz = dikkat(x, x, x)
_, maskeli = dikkat(x, x, x, causal_maske)

fig, eksenler = plt.subplots(1, 3, figsize=(17, 4.5))
for ax, (veri, baslik) in zip(eksenler, [(causal_maske.float(), "Causal maske (1 = bakabilir)"),
                                         (maskesiz, "Maskesiz dikkat ağırlıkları"),
                                         (maskeli, "Causal maskeli dikkat ağırlıkları")]):
    im = ax.imshow(veri, cmap="Blues", vmin=0, vmax=1)
    ax.set_xticks(range(T)); ax.set_xticklabels(tokenlar, rotation=30)
    ax.set_yticks(range(T)); ax.set_yticklabels(tokenlar)
    ax.set_title(baslik)
    for i in range(T):
        for j in range(T):
            ax.text(j, i, f"{veri[i, j]:.2f}", ha="center", va="center", fontsize=8,
                    color="white" if veri[i, j] > 0.5 else "black")
plt.tight_layout(); plt.show()

### Sızıntı testi

Maskenin gerçekten çalıştığını kanıtlamanın en iyi yolu: **gelecekteki** bir tokenı değiştirip **geçmişteki** çıktıların değişmediğini doğrulamak.

In [ ]:
x1 = torch.randn(1, 10, d)
x2 = x1.clone()
x2[0, 7] = torch.randn(d)                          # sadece 7. pozisyonu değiştir

maske = torch.tril(torch.ones(10, 10, dtype=torch.bool))
c1, _ = dikkat(x1, x1, x1, maske)
c2, _ = dikkat(x2, x2, x2, maske)
fark = (c1 - c2).abs().amax(-1)[0]

for t in range(10):
    print(f"pozisyon {t}: max |fark| = {fark[t]:.2e}  {'← değişti (beklenen)' if fark[t] > 1e-6 else ''}")

---
## 3) $O(n^2)$ bellek darboğazı ve FlashAttention

Standart dikkat, $n \times n$ boyutlu skor matrisini **GPU belleğinde (HBM)** oluşturur. Bellek ihtiyacı dizi uzunluğunun **karesiyle** büyür:

In [ ]:
satirlar = []
for n in [1_024, 4_096, 16_384, 65_536, 131_072]:
    bayt = n * n * 2                         # FP16: eleman başına 2 bayt
    satirlar.append({"dizi uzunluğu n": f"{n:,}",
                     "tek kafa, tek katman": f"{bayt / 2**20:,.0f} MB",
                     "32 kafa × 32 katman": f"{bayt * 32 * 32 / 2**30:,.0f} GB"})
pd.DataFrame(satirlar)

### FlashAttention (Dao vd., 2022) fikri

Asıl darboğaz çoğu zaman **hesaplama değil, bellek trafiği**: $n \times n$ matrisi yavaş HBM'ye yazıp tekrar okumak. FlashAttention:

1. Q, K, V'yi **bloklara (tile)** böler; her blok GPU'nun küçük ama çok hızlı **SRAM** belleğine sığar.
2. Skor matrisinin tamamını hiçbir zaman oluşturmaz; blok blok hesaplayıp hemen V ile çarpar.
3. Softmax'ı **online** hesaplar: her yeni blokta o ana kadarki satır maksimumunu $m$ ve normalizasyon toplamını $\ell$ günceller, önceki birikimi $e^{m_{\text{eski}} - m_{\text{yeni}}}$ ile düzeltir.

Sonuç **matematiksel olarak birebir aynıdır** (yaklaşık değil), ama bellek $O(n^2)$ yerine $O(n)$ olur ve çok daha hızlıdır.

$$
m^{(j)} = \max\!\left(m^{(j-1)}, \max_k s_k^{(j)}\right), \quad
\ell^{(j)} = e^{m^{(j-1)} - m^{(j)}}\,\ell^{(j-1)} + \sum_k e^{s_k^{(j)} - m^{(j)}}, \quad
O^{(j)} = e^{m^{(j-1)} - m^{(j)}}\,O^{(j-1)} + e^{S^{(j)} - m^{(j)}} V^{(j)}
$$

In [ ]:
def flash_attention_benzeri(Q, K, V, blok=128, causal=False):
    """Online softmax ile blok blok dikkat. n×n skor matrisi hiçbir zaman tam oluşturulmaz.
    Q, K, V: (n, d). Eğitim amaçlı saf PyTorch versiyonu; gerçek FlashAttention bunu tek bir CUDA kernel'inde yapar."""
    n, d = Q.shape
    olcek = 1.0 / math.sqrt(d)
    O = torch.empty_like(Q)
    for i in range(0, n, blok):
        q = Q[i:i + blok] * olcek
        b = q.size(0)
        m = torch.full((b, 1), float("-inf"))            # satır maksimumu
        l = torch.zeros(b, 1)                            # softmax paydası
        birikim = torch.zeros(b, d)                      # normalize edilmemiş çıktı
        for j in range(0, n, blok):
            if causal and j > i + b - 1:
                break                                    # tamamen gelecekte kalan bloklar atlanır
            s = q @ K[j:j + blok].T                      # (b, blok) — sadece küçük bir parça
            if causal:
                sorgu_idx = torch.arange(i, i + b).unsqueeze(1)
                anahtar_idx = torch.arange(j, j + s.size(1)).unsqueeze(0)
                s = s.masked_fill(anahtar_idx > sorgu_idx, float("-inf"))
            m_yeni = torch.maximum(m, s.max(-1, keepdim=True).values)
            p = torch.exp(s - m_yeni)
            duzeltme = torch.exp(m - m_yeni)             # eski birikimi yeni maksimuma göre ölçekle
            l = duzeltme * l + p.sum(-1, keepdim=True)
            birikim = duzeltme * birikim + p @ V[j:j + blok]
            m = m_yeni
        O[i:i + b] = birikim / l
    return O

n, d = 1024, 64
Q, K, V = torch.randn(n, d), torch.randn(n, d), torch.randn(n, d)

standart, _ = dikkat(Q, K, V)
standart_causal, _ = dikkat(Q, K, V, torch.tril(torch.ones(n, n, dtype=torch.bool)))
flash = flash_attention_benzeri(Q, K, V)
flash_causal = flash_attention_benzeri(Q, K, V, causal=True)
pytorch_sdpa = F.scaled_dot_product_attention(Q[None], K[None], V[None], is_causal=True)[0]

print(f"Standart vs blok-blok (maskesiz)    max fark: {(standart - flash).abs().max():.2e}")
print(f"Standart vs blok-blok (causal)      max fark: {(standart_causal - flash_causal).abs().max():.2e}")
print(f"Standart vs F.scaled_dot_product_attention: {(standart_causal - pytorch_sdpa).abs().max():.2e}")
print(f"\nStandart yöntemin ara bellek ihtiyacı: n×n = {n * n:,} eleman")
print(f"Blok yönteminin ara bellek ihtiyacı  : blok×blok = {128 * 128:,} eleman")

Pratikte FlashAttention'ı kendimiz yazmayız: PyTorch 2.0+ ile gelen `F.scaled_dot_product_attention`, donanıma göre otomatik olarak **FlashAttention**, **memory-efficient attention** veya standart çekirdeği seçer.

In [ ]:
def zamanla(f, tekrar=5):
    f()
    t0 = time.perf_counter()
    for _ in range(tekrar):
        f()
    return (time.perf_counter() - t0) / tekrar * 1000

cihaz = "cuda" if torch.cuda.is_available() else "cpu"
satirlar = []
for n in [512, 1024, 2048, 4096]:
    q = torch.randn(1, 8, n, 64, device=cihaz)
    k, v = torch.randn_like(q), torch.randn_like(q)
    maske = torch.tril(torch.ones(n, n, dtype=torch.bool, device=cihaz))
    satirlar.append({"n": n,
                     "elle yazılmış (ms)": zamanla(lambda: dikkat(q, k, v, maske)),
                     "F.scaled_dot_product_attention (ms)": zamanla(lambda: F.scaled_dot_product_attention(q, k, v, is_causal=True))})
    if cihaz == "cuda":
        torch.cuda.synchronize()

hiz = pd.DataFrame(satirlar).round(1)
hiz["hızlanma"] = (hiz["elle yazılmış (ms)"] / hiz["F.scaled_dot_product_attention (ms)"]).round(1).astype(str) + "×"
print("Cihaz:", cihaz)
hiz

---
## 4) RoPE — Rotary Positional Embeddings (Su vd., 2021)

Attention işlemi **sıra bilmez**: tokenların yerini değiştirirseniz çıktılar da aynı şekilde yer değiştirir. Konum bilgisini bir şekilde eklememiz gerekir.

RoPE, konumu toplamak yerine Q ve K vektörlerini konuma bağlı bir açıyla **döndürür**. Vektör ikişerli çiftlere ayrılır ve her çift $(x_{2i}, x_{2i+1})$ farklı bir frekansla döner:

$$
\begin{pmatrix} x'_{2i} \\ x'_{2i+1} \end{pmatrix} =
\begin{pmatrix} \cos m\theta_i & -\sin m\theta_i \\ \sin m\theta_i & \cos m\theta_i \end{pmatrix}
\begin{pmatrix} x_{2i} \\ x_{2i+1} \end{pmatrix},
\qquad \theta_i = 10000^{-2i/d}
$$

**Sihirli özellik:** $m$ konumundaki sorgu ile $n$ konumundaki anahtarın iç çarpımı **sadece aradaki farka** $(m - n)$ bağlıdır:

$$\langle R_m q,\; R_n k \rangle = \langle q,\; R_{n-m}\, k \rangle$$

Model mutlak konumu değil, **göreli mesafeyi** görür. Bu da daha uzun bağlamlara genelleme yapmayı kolaylaştırır (NTK / YaRN gibi bağlam uzatma yöntemleri RoPE frekanslarını ölçekler).

In [ ]:
def rope_frekanslari(d, maks_uzunluk, taban=10_000.0):
    theta = 1.0 / (taban ** (torch.arange(0, d, 2).float() / d))     # (d/2,)
    konum = torch.arange(maks_uzunluk).float()
    acilar = torch.outer(konum, theta)                                 # (L, d/2)
    return torch.cos(acilar), torch.sin(acilar)

def rope_uygula(x, cos, sin):
    """x: (..., L, d). Her (çift, tek) boyut çifti kendi frekansıyla döndürülür."""
    x_cift, x_tek = x[..., 0::2], x[..., 1::2]
    donmus_cift = x_cift * cos - x_tek * sin
    donmus_tek = x_cift * sin + x_tek * cos
    return torch.stack([donmus_cift, donmus_tek], dim=-1).flatten(-2)

d = 64
cos, sin = rope_frekanslari(d, 4096)
q, k = torch.randn(d), torch.randn(d)

def rope_konumda(v, m):
    return rope_uygula(v[None], cos[m:m + 1], sin[m:m + 1])[0]

print("Aynı göreli mesafe (3), farklı mutlak konumlar:")
for m, n in [(5, 2), (50, 47), (1000, 997), (3000, 2997)]:
    print(f"  q@{m:4d}, k@{n:4d} → q·k = {rope_konumda(q, m) @ rope_konumda(k, n):.5f}")

print(f"\nDöndürme normu korur: |q| = {q.norm():.4f}, |RoPE(q)| = {rope_konumda(q, 1234).norm():.4f}")

In [ ]:
fig, eksenler = plt.subplots(1, 2, figsize=(15, 4.5))

im = eksenler[0].imshow(cos[:128].T, aspect="auto", cmap="RdBu")
eksenler[0].set_xlabel("konum m"); eksenler[0].set_ylabel("frekans indeksi i")
eksenler[0].set_title("cos(m·θᵢ): düşük i hızlı döner, yüksek i yavaş döner")
plt.colorbar(im, ax=eksenler[0])

birler = torch.ones(d)
mesafeler = np.arange(0, 1024)
skorlar = [rope_konumda(birler, 0) @ rope_konumda(birler, int(r)) for r in mesafeler]
eksenler[1].plot(mesafeler, skorlar)
eksenler[1].set_xlabel("göreli mesafe |m − n|"); eksenler[1].set_ylabel("q·k")
eksenler[1].set_title("RoPE'nin uzun mesafe sönümü: uzak tokenlar arası benzerlik azalır")
plt.tight_layout(); plt.show()

---
## 5) RMSNorm (Zhang & Sennrich, 2019)

**LayerNorm:** ortalamayı çıkarır, standart sapmaya böler, öğrenilebilir ölçek $\gamma$ ve kaydırma $\beta$ uygular.

**RMSNorm:** ortalamayı çıkarmaz, sadece **karekök ortalama kare** (RMS) ile böler; $\beta$ yoktur.

$$\text{LayerNorm}(x) = \gamma \odot \frac{x - \mu}{\sqrt{\sigma^2 + \epsilon}} + \beta \qquad\qquad
\text{RMSNorm}(x) = \gamma \odot \frac{x}{\sqrt{\frac{1}{d}\sum_i x_i^2 + \epsilon}}$$

Deneyler, LayerNorm'un faydasının çoğunun **yeniden ölçekleme**den geldiğini, merkezlemenin pek katkı sağlamadığını gösterdi. RMSNorm daha az işlem yapar ve daha az parametreye sahiptir; LLaMA, Mistral, Qwen, Gemma hepsi RMSNorm kullanır. Modern modeller ayrıca normu alt katmandan **önce** uygular (**Pre-Norm**): $x + \text{Attn}(\text{Norm}(x))$. Bu derin ağlarda eğitimi çok daha kararlı yapar.

In [ ]:
class RMSNorm(nn.Module):
    def __init__(self, d, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.agirlik = nn.Parameter(torch.ones(d))

    def forward(self, x):
        rms = torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)
        return x * rms * self.agirlik

x = torch.randn(4, 16, 512) * 3 + 2
benim = RMSNorm(512)(x)
print(f"Bizim RMSNorm vs nn.RMSNorm max fark: {(benim - nn.RMSNorm(512, eps=1e-6)(x)).abs().max():.2e}")
print(f"RMSNorm sonrası RMS : {benim.pow(2).mean(-1).sqrt().mean():.3f}  (≈ 1)")
print(f"RMSNorm sonrası ort.: {benim.mean(-1).mean():.3f}  (sıfır değil! merkezleme yapılmıyor)")
print(f"LayerNorm sonrası ort.: {nn.LayerNorm(512)(x).mean(-1).mean():.3f}")
print(f"\nParametre: LayerNorm = {sum(p.numel() for p in nn.LayerNorm(512).parameters())}, "
      f"RMSNorm = {sum(p.numel() for p in RMSNorm(512).parameters())}")

---
## 6) SwiGLU (Shazeer, 2020)

Klasik FFN: $\text{FFN}(x) = W_2\,\text{ReLU}(W_1 x)$, gizli boyut genelde $4d$.

**SwiGLU** bir **kapı (gate)** mekanizması ekler: bir projeksiyon diğerinin hangi kısımlarının geçeceğine karar verir.

$$\text{SwiGLU}(x) = W_2\left(\text{SiLU}(W_1 x) \odot W_3 x\right), \qquad \text{SiLU}(z) = z \cdot \sigma(z)$$

Üç matris olduğu için parametre sayısını sabit tutmak adına gizli boyut $4d$ yerine $\frac{2}{3}\cdot 4d = \frac{8}{3}d$ seçilir. PaLM ve LLaMA makaleleri bu değişikliğin aynı parametre bütçesinde daha iyi perplexity verdiğini raporlamıştır.

In [ ]:
class KlasikFFN(nn.Module):
    def __init__(self, d, d_ff):
        super().__init__()
        self.w1, self.w2 = nn.Linear(d, d_ff, bias=False), nn.Linear(d_ff, d, bias=False)

    def forward(self, x):
        return self.w2(F.relu(self.w1(x)))


class SwiGLU(nn.Module):
    def __init__(self, d, d_ff):
        super().__init__()
        self.w1 = nn.Linear(d, d_ff, bias=False)    # kapı (gate) projeksiyonu
        self.w3 = nn.Linear(d, d_ff, bias=False)    # değer (up) projeksiyonu
        self.w2 = nn.Linear(d_ff, d, bias=False)    # aşağı (down) projeksiyon

    def forward(self, x):
        return self.w2(F.silu(self.w1(x)) * self.w3(x))

d = 4096
klasik = KlasikFFN(d, 4 * d)
swiglu = SwiGLU(d, int(8 * d / 3 / 256) * 256)          # LLaMA gibi 256'nın katına yuvarla
print(f"Klasik FFN (d_ff = {4 * d})  : {sum(p.numel() for p in klasik.parameters()) / 1e6:.1f} M parametre")
print(f"SwiGLU    (d_ff = {swiglu.w1.out_features}) : {sum(p.numel() for p in swiglu.parameters()) / 1e6:.1f} M parametre")

z = torch.linspace(-4, 4, 400)
plt.figure(figsize=(9, 4))
for isim, f in [("ReLU", F.relu), ("GELU", F.gelu), ("SiLU / Swish", F.silu)]:
    plt.plot(z, f(z), label=isim, lw=2)
plt.axhline(0, color="black", lw=0.5); plt.axvline(0, color="black", lw=0.5)
plt.title("Aktivasyon fonksiyonları: SiLU pürüzsüzdür ve negatif bölgede küçük bir 'çukur' içerir")
plt.legend(); plt.grid(alpha=0.3); plt.show()

---
## 7) Mixture of Experts (MoE) ve Yönlendirme (Routing)

Yoğun bir modelde **her token, her parametreden** geçer. MoE'de FFN katmanı $E$ adet **uzman** (expert) FFN ile değiştirilir ve küçük bir **yönlendirici (router)** her token için sadece en uygun $k$ uzmanı seçer:

$$
g(x) = \text{softmax}\big(\text{TopK}(x W_r,\; k)\big), \qquad
y = \sum_{e \in \text{TopK}} g_e(x)\, \text{Uzman}_e(x)
$$

| Model | Uzman sayısı | Token başına aktif | Toplam parametre | Aktif parametre |
|---|---|---|---|---|
| Mixtral 8×7B | 8 | 2 | ~47 B | ~13 B |
| DeepSeek-V3 | 256 (+1 paylaşılan) | 8 | 671 B | 37 B |
| Qwen3-30B-A3B | 128 | 8 | 30 B | 3 B |

**Avantaj:** Çok büyük bir modelin **bilgi kapasitesi**, küçük bir modelin **hesaplama maliyetiyle**.

**Sorun:** Router birkaç "favori" uzmana yığılırsa diğerleri hiç eğitilmez (**uzman çöküşü**). Çözüm: **yük dengeleme kaybı** (Switch Transformer):

$$\mathcal{L}_{\text{denge}} = E \cdot \sum_{e=1}^{E} f_e \cdot P_e$$

$f_e$: $e$ uzmanına gönderilen token oranı, $P_e$: router'ın $e$ uzmanına verdiği ortalama olasılık. Dağılım tamamen eşitken minimum değer olan 1'e ulaşır.

In [ ]:
class MoE(nn.Module):
    def __init__(self, d, d_ff, n_uzman=8, top_k=2):
        super().__init__()
        self.n_uzman, self.top_k = n_uzman, top_k
        self.yonlendirici = nn.Linear(d, n_uzman, bias=False)
        self.uzmanlar = nn.ModuleList([SwiGLU(d, d_ff) for _ in range(n_uzman)])

    def forward(self, x):
        B, T, D = x.shape
        duz = x.reshape(-1, D)                                        # (N, D)
        logit = self.yonlendirici(duz)                                # (N, E)
        olasilik = F.softmax(logit, dim=-1)
        topk_deger, topk_idx = logit.topk(self.top_k, dim=-1)         # (N, k)
        topk_agirlik = F.softmax(topk_deger, dim=-1)                  # seçilen k uzman arasında normalize

        cikti = torch.zeros_like(duz)
        for e, uzman in enumerate(self.uzmanlar):
            token_idx, slot = (topk_idx == e).nonzero(as_tuple=True)  # bu uzmana giden tokenlar
            if len(token_idx) == 0:
                continue
            cikti.index_add_(0, token_idx, topk_agirlik[token_idx, slot, None] * uzman(duz[token_idx]))

        # Switch Transformer yük dengeleme kaybı
        f = F.one_hot(topk_idx, self.n_uzman).float().sum(1).mean(0) / self.top_k
        P = olasilik.mean(0)
        self.denge_kaybi = self.n_uzman * (f * P).sum()
        self.son_atamalar = topk_idx.detach()
        return cikti.reshape(B, T, D)

d, d_ff = 128, 256
moe = MoE(d, d_ff, n_uzman=8, top_k=2)
tek_uzman = sum(p.numel() for p in moe.uzmanlar[0].parameters())
toplam = sum(p.numel() for p in moe.parameters())
print(f"Toplam parametre       : {toplam:,}")
print(f"Token başına aktif     : {2 * tek_uzman + moe.yonlendirici.weight.numel():,}  (2 uzman + router)")
print(f"Aktif / toplam oranı   : %{(2 * tek_uzman + moe.yonlendirici.weight.numel()) / toplam * 100:.0f}")

### Deney: uzman çöküşü ve yük dengeleme

Router'ı kasıtlı olarak bozuk başlatalım: tüm tokenlar 0 ve 1 numaralı uzmanlara gidiyor. Sonra yalnızca yük dengeleme kaybıyla router'ı eğitelim.

In [ ]:
torch.manual_seed(0)
x = torch.randn(16, 64, d) + 0.5                       # tüm tokenların ortak bir bileşeni var
moe = MoE(d, d_ff, n_uzman=8, top_k=2)
with torch.no_grad():
    moe.yonlendirici.weight[:2] = 0.1 * torch.ones(d)  # bu ortak bileşene güçlü tepki veren 2 "favori" uzman

def uzman_dagilimi():
    moe(x)
    return torch.bincount(moe.son_atamalar.flatten(), minlength=8).numpy(), moe.denge_kaybi.item()

once, once_kayip = uzman_dagilimi()
opt = torch.optim.Adam(moe.yonlendirici.parameters(), lr=1e-2)
for _ in range(300):
    moe(x)
    opt.zero_grad(); moe.denge_kaybi.backward(); opt.step()
sonra, sonra_kayip = uzman_dagilimi()

fig, eksenler = plt.subplots(1, 2, figsize=(13, 4), sharey=True)
eksenler[0].bar(range(8), once, color="tab:red")
eksenler[0].set_title(f"Önce: uzman çöküşü (denge kaybı = {once_kayip:.2f})")
eksenler[1].bar(range(8), sonra, color="tab:green")
eksenler[1].set_title(f"Sonra: dengeli yönlendirme (denge kaybı = {sonra_kayip:.2f})")
for ax in eksenler:
    ax.axhline(once.sum() / 8, color="gray", ls=":", label="ideal eşit dağılım")
    ax.set_xlabel("uzman"); ax.legend()
eksenler[0].set_ylabel("atanan token sayısı")
plt.tight_layout(); plt.show()

---
## 8) Hepsi bir arada: LLaMA tarzı Transformer bloğu

```
x ──┬──► RMSNorm ──► Q,K,V ──► RoPE(Q,K) ──► Causal SDPA ──► W_O ──► (+) ──┬──► RMSNorm ──► SwiGLU / MoE ──► (+) ──► çıktı
    └──────────────────────────────── artık bağlantı ───────────────────────┘└───────────── artık bağlantı ─────────┘
```

In [ ]:
class ModernDikkat(nn.Module):
    def __init__(self, d, n_kafa, maks_uzunluk=2048):
        super().__init__()
        self.n_kafa, self.d_kafa = n_kafa, d // n_kafa
        self.qkv = nn.Linear(d, 3 * d, bias=False)
        self.o = nn.Linear(d, d, bias=False)
        cos, sin = rope_frekanslari(self.d_kafa, maks_uzunluk)
        self.register_buffer("cos", cos, persistent=False)
        self.register_buffer("sin", sin, persistent=False)

    def forward(self, x):
        B, T, D = x.shape
        q, k, v = self.qkv(x).view(B, T, 3, self.n_kafa, self.d_kafa).permute(2, 0, 3, 1, 4)
        q = rope_uygula(q, self.cos[:T], self.sin[:T])         # konum bilgisi sadece Q ve K'ya
        k = rope_uygula(k, self.cos[:T], self.sin[:T])
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True)   # Flash/verimli çekirdek + causal maske
        return self.o(y.transpose(1, 2).reshape(B, T, D))


class LlamaBlok(nn.Module):
    def __init__(self, d=256, n_kafa=8, moe_kullan=False):
        super().__init__()
        self.norm1, self.norm2 = RMSNorm(d), RMSNorm(d)
        self.dikkat = ModernDikkat(d, n_kafa)
        d_ff = int(8 * d / 3)
        self.ffn = MoE(d, d_ff, n_uzman=4, top_k=2) if moe_kullan else SwiGLU(d, d_ff)

    def forward(self, x):
        x = x + self.dikkat(self.norm1(x))      # Pre-Norm + artık bağlantı
        x = x + self.ffn(self.norm2(x))
        return x


for moe_kullan in [False, True]:
    blok = LlamaBlok(moe_kullan=moe_kullan)
    x = torch.randn(2, 32, 256)
    y = blok(x)
    print(f"{'MoE' if moe_kullan else 'Dense'} blok: girdi {tuple(x.shape)} → çıktı {tuple(y.shape)}, "
          f"{sum(p.numel() for p in blok.parameters()):,} parametre")

# Uçtan uca causal kontrol: 20. tokenı değiştir, 0–19 arası çıktılar değişmemeli
blok = LlamaBlok().eval()
x1 = torch.randn(1, 32, 256); x2 = x1.clone(); x2[0, 20] += 1.0
with torch.no_grad():
    fark = (blok(x1) - blok(x2)).abs().amax(-1)[0]
print(f"\nPozisyon 0–19 max fark: {fark[:20].max():.2e}  |  pozisyon 20–31 max fark: {fark[20:].max():.2e}")

---
## Özet

| Bileşen | Çözdüğü problem | Fikir |
|---|---|---|
| **Scaled Dot-Product** | Tokenlar arası bilgi akışı | $\text{softmax}(QK^\top/\sqrt{d_k})V$ |
| **Multi-Head** | Tek bir ilişki tipiyle sınırlı kalmak | $h$ paralel dikkat, farklı alt uzaylar |
| **Causal Mask** | Eğitimde geleceği görme (kopya) | Üst üçgen = $-\infty$ |
| **FlashAttention** | $O(n^2)$ bellek, HBM trafiği | Bloklama + online softmax, birebir aynı sonuç |
| **RoPE** | Konum bilgisi, uzun bağlam | Q/K'yı döndür → iç çarpım göreli mesafeye bağlı |
| **RMSNorm** | Eğitim kararlılığı, hız | Merkezleme yok, sadece RMS ile ölçekleme |
| **SwiGLU** | FFN ifade gücü | Kapılı aktivasyon, $\frac{8}{3}d$ gizli boyut |
| **MoE** | Parametre ↑ ama hesaplama sabit | Router + top-k uzman + yük dengeleme kaybı |